In [ ]:
#Tubi Project File - Emerging Topics over time through X / Twitter
#Part 1: Twitter API Json Files Get

In [1]:
#Step 0: Libraries
import requests
import os
import json
import socket
import sys
import errno
import time
import sys
import pandas as pd
from io import StringIO
import numpy as np

In [2]:
#Step 0.5: Initialize data and counter
data = []
counter = 0

In [3]:
#Step 1: Bearer_Token
bearer_token = os.environ.get("xxxxxxxxxxxxxxxx")

In [4]:
#Step 2: bearer_oauth function
def bearer_oauth(r):
    """
    Method required by bearer token authentication.
    """

    r.headers["Authorization"] = f"xxxxxxxxxxxxxxxxx"
    r.headers["User-Agent"] = "v2FilteredStreamPython"
    return r

In [5]:
#Step 3: Get the Rules Function
def get_rules():
    response = requests.get(
        "https://api.twitter.com/2/tweets/search/stream/rules", auth=bearer_oauth
    )
    if response.status_code != 200:
        raise Exception(
            "Cannot get rules (HTTP {}): {}".format(response.status_code, response.text)
        )
    print(json.dumps(response.json()))
    return response.json()

In [6]:
#Step 4: Delete all the Rules Function, to clean the slate 
def delete_all_rules(rules):
    if rules is None or "data" not in rules:
        return None

    ids = list(map(lambda rule: rule["id"], rules["data"]))
    payload = {"delete": {"ids": ids}}
    response = requests.post(
        "https://api.twitter.com/2/tweets/search/stream/rules",
        auth=bearer_oauth,
        json=payload
    )
    if response.status_code != 200:
        raise Exception(
            "Cannot delete rules (HTTP {}): {}".format(
                response.status_code, response.text
            )
        )
    print(json.dumps(response.json()))

In [7]:
#Step 5: Add the New Rules Function
def set_rules(delete):
    # You can adjust the rules if needed
    sample_rules = [
        {"value": "\"military invasion\" OR \"military attack\" OR \"military clash\" OR \"military assault\" OR \"seize position\" lang: en", "tag": "Goldstein Negative"},
        {"value": "\"seize possessions\" OR \"nonmilitary destruction\" OR \"nonmilitary injury\" OR \"force mobilization\" OR \"force exercise\" lang: en", "tag": "Goldstein Negative"},
        {"value": "\"diplomatic recognition\" OR \"substantive agreement\" OR \"economic aid\" OR \"military assistance\" OR \"grant priviledge\" lang: en", "tag": "Goldstein Positive"},
        {"value": "\"suspend sanctions\" OR \"call truce\" OR \"material assistance\" OR \"endorse position\" OR \"verbal support\" lang: en", "tag": "Goldstein Positive"},
        {"value": "\"nuclear weapons\" OR \"nuclear weapon\" OR \"ballistic missile\" OR \"nuclear conflict\" OR \"nuclear attack\" lang: en", "tag": "Nuclear Threat"},
        {"value": "\"cyber attack\" OR \"cyber capabilities\" OR \"cyber defense\" OR \"cyber warfare\" OR \"cyber terrorism\" lang: en", "tag": "Cyber Warfare"},
        {"value": "\"war risk\" OR \"war fear\" OR \"military threat\" lang: en", "tag": "War Threats"},
        {"value": "\"oil crisis\" OR \"oil price\" OR \"petroleum exporting\" OR \"crude oil\" OR \"oil production\" lang: en", "tag": "Oil Supply Shock"},
        {"value": "\"China Sea\" OR \"China relations\" OR \"China trade\" OR \"Thucydides trap\" OR \"Taiwan sales\" lang: en", "tag": "US-China Relations"},
        {"value": "\"state terrorism\" OR \"counter terrorism\" OR \"terrorist attack\" OR \"political violence\" OR \"global terrorism\" lang: en", "tag": "Terrorism"},
        {"value": "\"geopolitical risk\" OR \"geopolitical concern\" OR \"geopolitical tension\" OR \"geopolitical uncertainty\" lang: en", "tag": "Geopolitical Risks"},
        {"value": "\"invasión militar\" OR \"ataque militar\" OR \"enfrentamiento militar\" OR \"asalto militar\" OR \"apoderamiento de la posición\" lang: es", "tag": "Goldstein Negative"},
        {"value": "\"apoderamiento de los bienes\" OR \"destrucción no militar\" OR \"daño no militar\" OR \"mobilización de fuerzas\" OR \"ejercicio de fuerzas\" lang: es", "tag": "Goldstein Negative"},
        {"value": "\"reconocimiento diplomático\" OR \"acuerdo sustancial\" OR \"ayuda económica\" OR \"asistencia militar\" OR \"otorgar privilegios\" lang: es", "tag": "Goldstein Positive"},
        {"value": "\"suspensión de sanciones\" OR \"tregua\" OR \"material de asistencia\" OR \"respaldo de posición\" OR \"apoyo verbal\" lang: es", "tag": "Goldstein Positive"},
        {"value": "\"armas nucleares\" OR \"arma nuclear\" OR \"misil balístico\" OR \"conflicto nuclear\" OR \"ataque nuclear\" lang: es", "tag": "Nuclear Threat"},
        {"value": "\"ciberataque\" OR \"capacidades cibernéticas\" OR \"defensa cibernética\" OR \"guerra cibernética\" OR \"ciberterrorismo\" lang: es", "tag": "Cyber Warfare"},
        {"value": "\"riesgo de guerra\" OR \"miedo a la guerra\" OR \"amenaza militar\" lang: es", "tag": "War Threats"},
        {"value": "\"crisis del petróleo\" OR \"precio del petróleo\" OR \"exportación del petróleo\" OR \"petróleo crudo\" OR \"producción del petróleo\" lang: es", "tag": "Oil Supply Shock"},
        {"value": "\"Mar de China\" OR \"relaciones de China\" OR \"comercio con China\" OR \"trampa de Tucídides\" OR \"ventas de Taiwan\" lang: es", "tag": "US-China Relations"},
        {"value": "\"estado terrorista\" OR \"lucha contra el terrorismo\" OR \"ataque terrorista\" OR \"violencia política\" OR \"terrorismo global\" lang: es", "tag": "Terrorism"},
        {"value": "\"riesgo geopolítico\" OR \"preocupación geopolítica\" OR \"tensión geopolítica\" OR \"incertidumbre geopolítica\" lang: es", "tag": "Geopolitical Risks"},
        {"value": "\"invasion militaire\" OR \"attaque militaire\" OR \"affrontement militaire\" OR \"assaut militaire\" OR \"saisir une position\" lang: fr", "tag": "Goldstein Negative"},
        {"value": "\"saisir des possessions\" OR \"destruction non militaire\" OR \"blessure non militaire\" OR \"mobilisation des forces\" OR \"exercice des forces\" lang: fr", "tag": "Goldstein Negative"},
        {"value": "\"reconnaissance diplomatique\" OR \"accord de fond\" OR \"aide économique\" OR \"aide militaire\" OR \"accorder un privilège\" lang: fr", "tag": "Goldstein Positive"},
        {"value": "\"suspendre les sanctions\" OR \"demander une trêve\" OR \"assistance matérielle\" OR \"soutenir une position\" OR \"soutien verbal\" lang: fr", "tag": "Goldstein Positive"},
        {"value": "\"armes nucléaires\" OR \"arme nucléaire\" OR \"missile balistique\" OR \"conflit nucléaire\" OR \"attaque nucléaire\" lang: fr", "tag": "Nuclear Threat"},
        {"value": "\"cyberattaque\" OR \"capacités cybernétiques\" OR \"cyberdéfense\" OR \"cyberguerre\" OR \"cyberterrorisme\" lang: fr", "tag": "Cyber Warfare"},
        {"value": "\"risque de guerre\" OR \"peur de la guerre\" OR \"menace militaire\" lang: fr", "tag": "War Threats"},
        {"value": "\"crise du pétrole\" OR \"prix du pétrole\" OR \"exportation de pétrole\" OR \"pétrole brut\" OR \"production de pétrole\" lang: fr", "tag": "Oil Supply Shock"},
        {"value": "\"Mer de Chine\" OR \"relations avec la Chine\" OR \"commerce Chinois\" OR \"piège de Thucydide\" OR \"ventes à Taiwan\" lang: fr", "tag": "US-China Relations"},
        {"value": "\"terrorisme d'État\" OR \"contre-terrorisme\" OR \"attentat terroriste\" OR \"violence politique\" OR \"terrorisme mondial\" lang: fr", "tag": "Terrorism"},
        {"value": "\"risque géopolitique\" OR \"préoccupation géopolitique\" OR \"tension géopolitique\" OR \"incertitude géopolitique\" lang: fr", "tag": "Geopolitical Risks"}, 
        {"value": "\"invasão militar\" OR \"ataque militar\" OR \"conflito militar\" OR \"investida militar\" OR \"tomar posição\" lang: pt", "tag": "Goldstein Negative"},
        {"value": "\"tomar os bens\" OR \"destruição não militar\" OR \"danos não militares\" OR \"mobilização da força\" OR \"exercício da força\" lang: pt", "tag": "Goldstein Negative"},
        {"value": "\"reconhecimento diplomático\" OR \"acordo substantivo\" OR \"ajuda econômica\" OR \"assistência militar\" OR \"conceder privilégios\" lang: pt", "tag": "Goldstein Positive"},
        {"value": "\"suspender sanções\" OR \"pedir trégua\" OR \"assistência material\" OR \"endossar posição\" OR \"apoio verbal\" lang: pt", "tag": "Goldstein Positive"},
        {"value": "\"armas nucleares\" OR \"arma nuclear\" OR \"míssil balístico\" OR \"conflito nuclear\" OR \"ataque nuclear\" lang: pt", "tag": "Nuclear Threat"},
        {"value": "\"ataque cibernético\" OR \"habilidades cibernéticas\" OR \"defesa cibernética\" OR \"guerra cibernética\" OR \"terrorismo cibernético\" lang: pt", "tag": "Cyber Warfare"},
        {"value": "\"risco de guerra\" OR \"medo de guerra\" OR \"ameaça militar\" lang: pt", "tag": "War Threats"},
        {"value": "\"crise do petróleo\" OR \"preço do petróleo\" OR \"exportação de petróleo\" OR \"petróleo bruto\" OR \"produção de petróleo\" lang: pt", "tag": "Oil Supply Shock"},
        {"value": "\"Mar da China\" OR \"relações com a China\" OR \"comércio da China\" OR \"armadilha de Tucídides\" OR \"vendas em Taiwan\" lang: pt", "tag": "US-China Relations"},
        {"value": "\"terrorismo de estado\" OR \"contraterrorismo\" OR \"ataque terrorista\" OR \"violência política\" OR \"terrorismo global\" lang: pt", "tag": "Terrorism"},
        {"value": "\"risco geopolítico\" OR \"preocupação geopolítica\" OR \"tensão geopolítica\" OR \"incerteza geopolítica\" lang: pt", "tag": "Geopolitical Risks"},
        {"value": "\"غزو عسكري\" OR \"هجوم عسكري\" OR \"صدام عسكري\" OR \"اعتداء عسكري\" OR \"تولي المنصب\" lang: ar", "tag": "Goldstein Negative"},
        {"value": "\"الاستيلاء على الممتلكات\" OR \"تدمير غير عسكري\" OR \"إصابة غير عسكرية\" OR \"حشد القوات\" OR \"تدريب القوات\" lang: ar", "tag": "Goldstein Negative"},
        {"value": "\"الاعتراف الدبلوماسي\" OR \"اتفاق جوهري\" OR \"عون اقتصادي\" OR \"مساعدة عسكرية\" OR \"منح الامتياز\" lang: ar", "tag": "Goldstein Positive"},
        {"value": "\"تعليق الجزاءات\" OR \"عقد هدنة\" OR \"المساعدة المادية\" OR \"تأييد الموقف\" OR \"دعم شفوي\" lang: ar", "tag": "Goldstein Positive"},
        {"value": "\"الأسلحة النووية\" OR \"سلاح نووي\" OR \"القذائف التسيارية\" OR \"راع نووي\" OR \"هجوم نووي\" lang: ar", "tag": "Nuclear Threat"},
        {"value": "\"الهجوم الإلكتروني\" OR \"القدرات الإلكترونية\" OR \"الدفاع الإلكتروني\" OR \"الحرب الإلكترونية\" OR \"إرهاب إلكتروني\" lang: ar", "tag": "Cyber Warfare"},
        {"value": "\"مخاطر الحرب\" OR \"مخاوف الحرب\" OR \"التهديد العسكر\" lang: ar", "tag": "War Threats"},
        {"value": "\"أزمة النفط\" OR \"سعر النفط\" OR \"تصدير البترول\" OR \"النفط الخام\" OR \"إنتاج النفط\" lang: ar", "tag": "Oil Supply Shock"},
        {"value": "\"بحر الصين\" OR \"العلاقات الصينية\" OR \"تجارة الصين\" OR \"مصيدة توسيديس\" OR \"مبيعات تايوان\" lang: ar", "tag": "US-China Relations"},
        {"value": "\"إرهاب الدولة\" OR \"مكافحة الإرهاب\" OR \"هجوم إرهابي\" OR \"عنف سياسي\" OR \"إرهاب عالمي\" lang: ar", "tag": "Terrorism"},
        {"value": "\"مخاطر جغرافية سياسية\" OR \"مخاوف جغرافية سياسية\" OR \"توترات جغرافية سياسية\" OR \"شكوك جغرافية سياسية\" lang: ar", "tag": "Geopolitical Risks"},
        {"value": "\"軍による侵攻\" OR \"軍による攻撃\" OR \"軍の衝突\" OR \"軍による暴行\" OR \"奪取地点\" lang: ja", "tag": "Goldstein Negative"},
        {"value": "\"所持品を奪取\" OR \"軍以外による破壊行為\" OR \"軍以外の負傷\" OR \"部隊の動員\" OR \"部隊の訓練\" lang: ja", "tag": "Goldstein Negative"},
        {"value": "\"外交的認識\" OR \"重要な合意\" OR \"経済支援\" OR \"軍の支援\" OR \"権限を付与\" lang: ja", "tag": "Goldstein Positive"},
        {"value": "\"制裁を一時中止\" OR \"休戦を呼びかける\" OR \"物的援助\" OR \"立場を支持\" OR \"言葉での支援\" lang: ja", "tag": "Goldstein Positive"},
        {"value": "\"核兵器\" OR \"核兵器\" OR \"弾道ミサイル\" OR \"核紛争\" OR \"核攻撃\" lang: ja", "tag": "Nuclear Threat"},
        {"value": "\"サイバー攻撃\" OR \"サイバー能力\" OR \"サイバー防衛\" OR \"サイバー空間での交戦\" OR \"サイバーテロ\" lang: ja", "tag": "Cyber Warfare"},
        {"value": "\"戦争のリスク\" OR \"戦争の恐怖\" OR \"軍の脅迫\" lang: ja", "tag": "War Threats"},
        {"value": "\"石油危機\" OR \"原油価格\" OR \"石油の輸出\" OR \"原油\" OR \"原油生産\" lang: ja", "tag": "Oil Supply Shock"},
        {"value": "\"シナ海\" OR \"中国との関係\" OR \"中国の貿易\" OR \"トゥキュディデスの罠\" OR \"台湾の売り上げ\" lang: ja", "tag": "US-China Relations"},
        {"value": "\"国家主導のテロ\" OR \"テロ対策\" OR \"テロ攻撃\" OR \"政治的暴力\" OR \"グローバルテロリズム\" lang: ja", "tag": "Terrorism"},
        {"value": "\"地政治学上のリスク\" OR \"地政治学上の懸念\" OR \"地政治学上の緊張\" OR \"地政治学上の不確定要素\" lang: ja", "tag": "Geopolitical Risks"},
        {"value": "\"군사적 침공\" OR \"군사적 공격\" OR \"군사적 충돌\" OR \"군사적 공격\" OR \"진지 점령\" lang: ko", "tag": "Goldstein Negative"},
        {"value": "\"소유물 탈취\" OR \"비군사적 파괴\" OR \"비군사적 부상\" OR \"무력 동원\" OR \"무력 훈련\" lang: ko", "tag": "Goldstein Negative"},
        {"value": "\"외교적 인정\" OR \"실질적 합의\" OR \"경제 원조\" OR \"군사 지원\" OR \"권한 부여\" lang: ko", "tag": "Goldstein Positive"},
        {"value": "\"제재 유예\" OR \"휴전 선언\" OR \"물적 지원\" OR \"입장 지지\" OR \"구두 지원\" lang: ko", "tag": "Goldstein Positive"},
        {"value": "\"핵무기\" OR \"핵무기\" OR \"탄도 미사일\" OR \"핵 분쟁\" OR \"핵공격\" lang: ko", "tag": "Nuclear Threat"},
        {"value": "\"사이버 공격\" OR \"사이버 능력\" OR \"사이버 방어\" OR \"사이버 워페이스\" OR \"사이버 테러\" lang: ko", "tag": "Cyber Warfare"},
        {"value": "\"전쟁 위험\" OR \"전쟁 공포\" OR \"군사적 위협\" lang: ko", "tag": "War Threats"},
        {"value": "\"석유 파동\" OR \"유가\" OR \"석유 수출\" OR \"원유\" OR \"석유 생산\" lang: ko", "tag": "Oil Supply Shock"},
        {"value": "\"중국해\" OR \"중국 관계\" OR \"중국 무역\" OR \"투키디데스 함정\" OR \"대만 판매\" lang: ko", "tag": "US-China Relations"},
        {"value": "\"국가 테러\" OR \"대테러\" OR \"테러 공격\" OR \"정치적 폭력\" OR \"글로벌 테러\" lang: ko", "tag": "Terrorism"},
        {"value": "\"지정학적 위험\" OR \"지정학적 관심사\" OR \"지정학적 긴장\" OR \"지정학적 불확실성\" lang: ko", "tag": "Geopolitical Risks"},    
    ]
    payload = {"add": sample_rules}
    response = requests.post(
        "https://api.twitter.com/2/tweets/search/stream/rules",
        auth=bearer_oauth,
        json=payload,
    )
    if response.status_code != 201:
        raise Exception(
            "Cannot add rules (HTTP {}): {}".format(response.status_code, response.text)
        )
    print(json.dumps(response.json()))

In [8]:
#Merge Dictionaries Function - Helper Function
#https://www.geeksforgeeks.org/python-merging-two-dictionaries/
def Merge(dict1, dict2):
    res = {**dict1, **dict2}
    return res

In [9]:
#Step 6: Get Stream API Started Function
def get_stream(set):
    global data, counter
    response = requests.get(
        "https://api.twitter.com/2/tweets/search/stream?tweet.fields=created_at", auth=bearer_oauth, stream=True,
    )
    print(response.status_code)
    if response.status_code != 200:
        raise Exception(
            "Cannot get stream (HTTP {}): {}".format(
                response.status_code, response.text
            )
        )
    for response_line in response.iter_lines():
        if response_line:
            json_response = json.loads(response_line)
            print(json.dumps(json_response, indent = 4, sort_keys = True))
            data_dic = json_response['data']
            tag_str = json_response['matching_rules'][0]["tag"]
            tag_dict_temp = {"tag": tag_str}
            #https://favtutor.com/blogs/string-to-dict-python
            full_Merge_dic = Merge(data_dic, tag_dict_temp)
            data.append(full_Merge_dic)
            #Store every 25 tweets
            if len(data) % 100 == 0:
                print('storing data')
                temp = json.dumps(data)
                pd.read_json(StringIO(temp)).to_json(f'/Users/johnc.burns/Documents/Documents/PhD Year Two/Mockup 9/foo5/json_file/tw_example_{counter}.json', orient='records')
                data = [] #Resets the data frame
                counter += 1 

In [10]:
#Main function 2 test
#https://docs.python.org/3/library/signal.html#note-on-sigpipe
def main_2(): 
        rules = get_rules()
        delete = delete_all_rules(rules)
        set = set_rules(delete)
        get_stream(set)

In [11]:
#Step 9 Alt: Reconnection automatically
#https://stackoverflow.com/questions/32267540/jupyter-notebook-how-to-relaunch-all-cells-above-when-a-crash-occurs
if __name__ == "__main__":
    try:
        main_2()
    except:
        display(Javascript('IPython.notebook.execute_cell_range(IPython.notebook.get_selected_index()+1,'+
                           ' IPython.notebook.get_selected_index()+2)'))


{"data": [{"id": "1508993278313697284", "value": "\"geopolitical risk\" OR \"geopolitical concern\" OR \"geopolitical tension\" OR \"geopolitical uncertainty\" lang: en", "tag": "Geopolitical Risks"}, {"id": "1508993278313697285", "value": "\"\u062a\u0647\u062f\u064a\u062f \u0625\u0631\u0647\u0627\u0628\u064a\" OR \"\u062a\u0648\u0639\u0651\u062f \u0625\u0631\u0647\u0627\u0628\u064a\" lang: ar", "tag": "Terrorist Threats"}, {"id": "1508993278313697286", "value": "\"inicio de la guerra\" OR \"estallido de la guerra\" OR \"Guerra de escalas\" OR \"guerra de escalada\" lang: es", "tag": "War Acts"}, {"id": "1508993278313697287", "value": "\"d\u00e9but de guerre\" OR \"d\u00e9clenchement de la guerre\" OR \"d\u00e9clencher la guerre\" OR \"intensification de la guerre\" lang: fr", "tag": "War Acts"}, {"id": "1508993278313697288", "value": "\"risco de guerra\" OR \"medo de guerra\" OR \"amea\u00e7a militar\" lang: pt", "tag": "War Threats"}, {"id": "1508993278313697289", "value": "\"\u6226\

200
{
    "data": {
        "created_at": "2022-03-30T02:29:36.000Z",
        "id": "1508994826146131976",
        "text": "@Donevita1 https://t.co/a3vS5hBRgP\n\nAyude con un RT usted que es famoso Don Eva... \ud83d\ude4f"
    },
    "matching_rules": [
        {
            "id": "1508994838515134475",
            "tag": "Goldstein Positive"
        }
    ]
}
{
    "data": {
        "created_at": "2022-03-30T02:29:39.000Z",
        "id": "1508994836736532487",
        "text": "RT @JPN_LISA: \uff08\u30d5\u30e9\u30f3\u30af\u30d5\u30eb\u30c8\u65b0\u805e\u306e\u8a18\u4e8b\u3088\u308a\uff09\u30a6\u30af\u30e9\u30a4\u30ca\u6226\u4e89\u52c3\u767a\u76f4\u5f8c\u3001\u30a6\u30af\u30e9\u30a4\u30ca\u304c\u30c9\u30a4\u30c4\u306b\u652f\u63f4\u3092\u6c42\u3081\u305f\u6642\u3001\u300c\u30ad\u30a8\u30d5\u9665\u843d\u307e\u3067\u50c5\u304b\u306a\u6642\u9593\u3057\u304b\u306a\u3044\u306e\u3067\u652f\u63f4\u3059\u308b\u610f\u5473\u304c\u306a\u3044\u300d\u3068\u8a00\u3063\u305f\u306e\u306f\u30ea\u30f3\u30c

{
    "data": {
        "created_at": "2022-03-30T02:31:16.000Z",
        "id": "1508995243420520449",
        "text": "RT @NomuraDirect: (+) Geopolitical risk: \u0e19\u0e32\u0e22\u0e40\u0e21\u0e1f\u0e25\u0e38\u0e15 \u0e04\u0e32\u0e27\u0e39\u0e42\u0e0b\u0e01\u0e25\u0e39 \u0e23\u0e21\u0e27.\u0e15\u0e48\u0e32\u0e07\u0e1b\u0e23\u0e30\u0e40\u0e17\u0e28\u0e15\u0e38\u0e23\u0e01\u0e35\u0e01\u0e25\u0e48\u0e32\u0e27\u0e27\u0e48\u0e32 \u0e01\u0e32\u0e23\u0e40\u0e08\u0e23\u0e08\u0e32\u0e2a\u0e31\u0e19\u0e15\u0e34\u0e20\u0e32\u0e1e\u0e23\u0e30\u0e2b\u0e27\u0e48\u0e32\u0e07\u0e23\u0e31\u0e2a\u0e40\u0e0b\u0e35\u0e22\u0e41\u0e25\u0e30\u0e22\u0e39\u0e40\u0e04\u0e23\u0e19\u0e17\u0e35\u0e48\u0e01\u0e23\u0e38\u0e07\u0e2d\u0e34\u0e2a\u2026"
    },
    "matching_rules": [
        {
            "id": "1508994838515134487",
            "tag": "Geopolitical Risks"
        }
    ]
}
{
    "data": {
        "created_at": "2022-03-30T02:31:16.000Z",
        "id": "1508995243340967943",
        "text": "RT @Gusta

{
    "data": {
        "created_at": "2022-03-30T02:33:01.000Z",
        "id": "1508995685366083591",
        "text": "RT @ALVARITRIP: Por amor a Dios necesito ayuda econ\u00f3mica para salvar la vida de el ser que me ha sacado del suicidio ay\u00fadame dando retweet\u2026"
    },
    "matching_rules": [
        {
            "id": "1508994838515134475",
            "tag": "Goldstein Positive"
        }
    ]
}
{
    "data": {
        "created_at": "2022-03-30T02:33:17.000Z",
        "id": "1508995751422181377",
        "text": "RT @ALVARITRIP: Por amor a Dios necesito ayuda econ\u00f3mica para salvar la vida de el ser que me ha sacado del suicidio ay\u00fadame dando retweet\u2026"
    },
    "matching_rules": [
        {
            "id": "1508994838515134475",
            "tag": "Goldstein Positive"
        }
    ]
}
{
    "data": {
        "created_at": "2022-03-30T02:33:18.000Z",
        "id": "1508995754987343875",
        "text": "RT Masivo\nAyuda https://t.co/znhbv9iBEh"
    },

{
    "data": {
        "created_at": "2022-03-30T02:35:07.000Z",
        "id": "1508996212866707457",
        "text": "RT @s_w_s_m: \u300c\u529b\u306b\u3088\u308b\u73fe\u72b6\u5909\u66f4\u306f\u8a31\u3055\u306a\u3044\u3057\u540c\u76df\u56fd\u306f\u5b88\u308b\u3051\u3069\u3001\u3058\u3083\u3042\u540c\u76df\u95a2\u4fc2\u306e\u306a\u3044\u4ed6\u56fd\u306e\u9818\u571f\u304c\u7372\u3089\u308c\u308b\u304b\u3089\u3068\u7b2c\u4e09\u6b21\u5927\u6226\u3084\u6838\u6226\u4e89\u306e\u30ea\u30b9\u30af\u3092\u627f\u77e5\u3067\u6838\u6301\u3063\u305f\u6a29\u5a01\u4e3b\u7fa9\u9663\u55b6\u56fd\u3092\u6bb4\u308c\u308b\u306e\u304b\uff1f\u300d\u3063\u3066\u845b\u85e4\u306f\u5f37\u304f\u3001\u56fd\u3072\u3068\u3064\u6d88\u3048\u308b\u304b\u3069\u3046\u304b\u306e\u702c\u6238\u969b\u306b\u3042\u3063\u3066\u3059\u3089\u7c73\u82f1\u306e\u884c\u52d5\u3092\u7e1b\u3063\u3066\u308b\u308f\u3051\u3060\u3057\u2026"
    },
    "matching_rules": [
        {
            "id": "1508994838515134491",
            "tag": "W

...

{
    "data": {
        "created_at": "2022-03-30T03:38:30.000Z",
        "id": "1509012165050912769",
        "text": "@JustinTrudeau @ZelenskyyUa LMAO Like recognises like \ud83e\udd21\ud83c\udf0f \n\nDid you ask Z how he got away with jailing opposition politicians, banning opposition media and shelling his own civilians for the past 8 years? Have you told him about your horses trampling unarmed citizens? Asking for democracy https://t.co/vySK62jSLp"
    },
    "matching_rules": [
        {
            "id": "1508994838515134466",
            "tag": "Goldstein Positive"
        }
    ]
}
{
    "data": {
        "created_at": "2022-03-30T03:38:35.000Z",
        "id": "1509012182310297600",
        "text": "RT @Jorge_Elbaum: Fragmento del debate \n\nEl inicio de la guerra fue en 2014 con el asesinato de 42 personas en Odessa\n\nLa limpieza \u00e9tnica c\u2026"
    },
    "matching_rules": [
        {
            "id": "1508994838515134481",
            "tag": "War Acts"
        }
    

{
    "data": {
        "created_at": "2022-03-30T03:40:44.000Z",
        "id": "1509012726735335424",
        "text": "RT @JPN_LISA: \uff08\u30d5\u30e9\u30f3\u30af\u30d5\u30eb\u30c8\u65b0\u805e\u306e\u8a18\u4e8b\u3088\u308a\uff09\u30a6\u30af\u30e9\u30a4\u30ca\u6226\u4e89\u52c3\u767a\u76f4\u5f8c\u3001\u30a6\u30af\u30e9\u30a4\u30ca\u304c\u30c9\u30a4\u30c4\u306b\u652f\u63f4\u3092\u6c42\u3081\u305f\u6642\u3001\u300c\u30ad\u30a8\u30d5\u9665\u843d\u307e\u3067\u50c5\u304b\u306a\u6642\u9593\u3057\u304b\u306a\u3044\u306e\u3067\u652f\u63f4\u3059\u308b\u610f\u5473\u304c\u306a\u3044\u300d\u3068\u8a00\u3063\u305f\u306e\u306f\u30ea\u30f3\u30c8\u30ca\u30fc\u8ca1\u52d9\u5927\u81e3\u3060\u3063\u305f\u6a21\u69d8\u3002\u4ee5\u524d\u306f\u30d9\u30a2\u30dc\u30c3\u30af\u5916\u52d9\u5927\u81e3\u304c\u7591\u308f\u308c\u3066\u3044\u305f\u3002"
    },
    "matching_rules": [
        {
            "id": "1508994838515134474",
            "tag": "War Acts"
        }
    ]
}
{
    "data": {
        "created_at": 

{
    "data": {
        "created_at": "2022-03-30T03:42:35.000Z",
        "id": "1509013192583913484",
        "text": "RT @enlacejudio: El primer ministro de Israel manifest\u00f3 en un video su reacci\u00f3n tras el atentado terrorista en Bnei Brak que dej\u00f3 5 muertos\u2026"
    },
    "matching_rules": [
        {
            "id": "1508994838515134489",
            "tag": "Terrorist Threats"
        }
    ]
}
{
    "data": {
        "created_at": "2022-03-30T03:42:37.000Z",
        "id": "1509013197445120003",
        "text": "RT @Diego_Molano: A \u2018Jhon Mechas\u2019, responsable del atentado terrorista en Bogot\u00e1 contra ni\u00f1os, ni\u00f1as y adultos, lo neutralizaremos est\u00e9 don\u2026"
    },
    "matching_rules": [
        {
            "id": "1508994838515134489",
            "tag": "Terrorist Threats"
        }
    ]
}
{
    "data": {
        "created_at": "2022-03-30T03:42:42.000Z",
        "id": "1509013220794806276",
        "text": "RT @GustavoRugeles: Gus

{
    "data": {
        "created_at": "2022-03-30T03:45:26.000Z",
        "id": "1509013908623073287",
        "text": "RT @aoyamakoba: @sofimari21 \ud83d\udd35\u30ed\u30b7\u30a2\u56fd\u5185\u306e\u30c1\u30a7\u30c1\u30a7\u30f3\u5171\u548c\u56fd\u304b\u3089\u30de\u30ea\u30a6\u30dd\u30ea\u5e02\u8857\u6226\u306b\u3084\u3063\u3066\u6765\u305f\u30ed\u30b7\u30a2\u8ecd\u306e\u652f\u63f4\u968a\u304b\uff1f\n\u3000\u76ee\u7684\u306f\u30a6\ud83c\uddfa\ud83c\udde6\u8ecd\u30a2\u30be\u30d5\u5927\u968a\u30cd\u30aa\u30ca\u30c1\u3092\u4e00\u6383\u3057\u30de\u30ea\u30a6\u30dd\u30ea\u5e02\u6c11\u3092\u907f\u96e3\u3055\u305b\u308b\u70ba\u3068\u805e\u3044\u3066\u3044\u307e\u3059\u3002"
    },
    "matching_rules": [
        {
            "id": "1508994838515134484",
            "tag": "Goldstein Positive"
        }
    ]
}


KeyboardInterrupt: 

In [ ]:
#Part 2: Emerging Topics over Time (Only English)

In [ ]:
#Step 1: Libraries
#The Glob Libraries
#https://stackoverflow.com/questions/57067551/how-to-read-multiple-json-files-into-pandas-dataframe
#/Users/johnc.burns/Documents/Documents/PhD Year Two/Mockup 9/foo5/json_file
import os
import json
import pandas as pd
import numpy as np
import glob
pd.set_option('display.max_columns', 500)
import csv

#Lang Detect Libraries
#Get the language labels for the dataset
#Enforce consistent results 
from langdetect import DetectorFactory
DetectorFactory.seed = 0
#Import the detect function
from langdetect import detect

#Topics over Time Libraries
import numpy as np
from pprint import pprint
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
import gensim.test.utils
import spacy
import en_core_web_sm
import nltk
from nltk.corpus import stopwords
import tqdm

#Pretty Printing Library
from pprint import pprint

#Making the Plot Library
#Graphing in Jupiter Notebook
import matplotlib.pyplot as plt
%matplotlib inline 

#Translation Library
from deep_translator import GoogleTranslator

In [ ]:
#Step 2: Constants
#Set the path to the json 
path_to_json = "/Users/johnc.burns/Documents/Documents/PhD Year Three/My Paper 5/foo9/json_file"

#Set Counter variable
counter_tm = 0

In [ ]:
#Step 3: Functions
#Part 1: Get all files from Json_file output
def all_files(path_to_json):
    json_pattern = os.path.join(path_to_json, '*.json')
    file_list = glob.glob(json_pattern)
    #Import the jsons and convert them to pandas dataframes
    dfs = []
    for file in file_list:
        data = pd.read_json(file)
        dfs.append(data)
    #Concat the individual data frames into one dataframe
    full_tm = pd.concat(dfs, ignore_index = True)
    #Reset Index
    ftm2 = full_tm.reset_index(drop = True)
    return ftm2

In [ ]:
#Part 2: Language detect Function
def lang_detect_2(text):
    detectanswer = detect(text)
    return detectanswer

In [ ]:
#Part 3: Set up the language detection loop Function
def lang_loop(df):
    #Get Length of dataframe
    lendf = len(df)
    #Create new column with string
    df['lang'] = ""
    #Use the language Detect function to get the language of the text
    for i in range(0, lendf):
        try:
            df["lang"][i] = lang_detect_2(df["text"][i])
        except LangDetectException:
            df["lang"][i] = "und"
    return df

In [ ]:
#Part 4: Break into English Tweets
def English_Tweets_Func(df_gt):
    English_Tweets = df_gt[df_gt['lang'] == "en"]
    English_Tweets.reset_index(drop = True, inplace = True)
    English_Tweets['TweetNumber'] = np.arange(len(English_Tweets))
    return English_Tweets

In [ ]:
#Part 5: Create the Date Function - for time processing
def date_func(df):
    df["Date"] = ""
    lendf = len(df)
    for i in range(lendf):
        df["Date"][i] = str(df["created_at"][i])
    return df

In [ ]:
#Step 6: Create the Year Function
def year_func(df):
    df["Year"] = ""
    lendf = len(df)
    for i in range(lendf):
        df["Year"][i] = int(df["Date"][i][0:4]) 
    return df

In [ ]:
#Step 7: Create the Month Function
def month_func(df):
    df["Month"] = ""
    lendf = len(df)
    for i in range(lendf):
        df["Month"][i] = int(df["Date"][i][5:7])
    return df

In [ ]:
#Step 8: Create the Day Function
def day_func(df):
    df["Day"] = ""
    lendf = len(df)
    for i in range(lendf):
        df["Day"][i] = int(df["Date"][i][8:10])
    return df

In [ ]:
#Step 9: Create the Hour Function
def hour_func(df):
    df["Hour"] = ""
    lendf = len(df)
    for i in range(lendf):
        df["Hour"][i] = int(df["Date"][i][11:13])
    return df

In [ ]:
#Step 10: Create the Minute Function
def minute_func(df):
    df["Minute"] = ""
    lendf = len(df)
    for i in range(lendf):
        df["Minute"][i] = int(df["Date"][i][14:16])
    return df

In [ ]:
#Step 11: Create the Second Function
def second_func(df):
    df["Second"] = ""
    lendf = len(df)
    for i in range(lendf):
        df["Second"][i] = int(df["Date"][i][17:19])
    return df

In [ ]:
#Step 12: Sort Chronologically Function Sort by the Year, Month, Day, Hour, Minute, Second
def sort_chrono(df):
    df2 = df.sort_values(by = ['Year', 'Month', 'Day', 'Hour', 'Minute', 'Second'], ascending = ['False', 'False', 'False', 'False', 'False', 'False'], na_position = 'first')
    df3 = df2.reset_index(drop = True)
    return df3

In [ ]:
#Step 13: English Stop Words Vector
def stopwords_en_func():
    stop_words_en = stopwords.words('english')
    custom_stop_words = ["http", "https", "co", "com", "app", "go", "amp", "RT"]
    final_stop_words_en = stop_words_en + custom_stop_words
    return final_stop_words_en

In [ ]:
#Step 14: Remove Stop Words English
def stopwords_en(texts, final_stop_words_en):
    return[[word for word in simple_preprocess(str(doc)) if word not in final_stop_words_en] for doc in texts]

In [ ]:
#Step 15: Make bigrams of the words
#Make sure do call this function 7 times for each of the 7 languages
def bigrams(texts):
    bigram = gensim.models.Phrases(texts, min_count = 5, threshold = 100)
    bigram_mod = gensim.models.phrases.Phraser(bigram)
    return [bigram_mod[doc] for doc in texts]

In [ ]:
#Step 16: Turn the words into lemmas English (Cutting off the endings of words (ex: -ed, -ing) for better groupings
#of the topic modeling process
def data_lemmatization_en(texts, allowed_postags = ['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    nlp = spacy.load('en_core_web_sm', disable = ['parser', 'ner'])
    nlp.max_length = 15000000
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent))
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [ ]:
#Step 17: Create topic_id numbers, change dynamically
#https://www.delftstack.com/howto/python/python-list-from-1-to-n/
#topic_id = [0, 1, 2, 3, 4, 5, 6, 7, 8]
def createList(n):
    lst1 = []
    for i in range(n):
        lst1.append(i)
    return(lst1)

In [ ]:
#Step 18: Detour to focus on hyperparameter tunning of the LDA Model
#https://towardsdatascience.com/evaluate-topic-model-in-python-latent-dirichlet-allocation-lda-7d57484bb5d0
#First build the function to test the hyperparameters of the number of topics (k), the document - topic density
#(alpha) and the Word - topic density (beta), we get the coherence score for each model using the 'c_v' which is 
#measure is based on a sliding window, one-set segmentation of the top words and an indirect confirmation 
#measure that uses normalized pointwise mutual information (NPMI) and the cosine similarity

def compute_coherence_values(corpus, texts, dictionary, k, a, b):

    lda_model_cv = gensim.models.LdaMulticore(corpus = corpus,
                                         id2word = dictionary,
                                         num_topics = k,
                                         random_state = 101,
                                         chunksize = 100,
                                         passes = 10,
                                         alpha = a,
                                         eta = b,
                                          per_word_topics = True,
                                          minimum_probability = 0)
    
    coherence_model_lda = CoherenceModel(model = lda_model_cv, texts = texts, 
                                         dictionary = dictionary, coherence = 'c_v')
    
    return coherence_model_lda.get_coherence()

In [ ]:
#Step 19: LDA Hyperparameter Finding function for English
#https://stackoverflow.com/questions/60087463/valueerror-stop-argument-for-islice-must-be-none-or-an-integer-0-x-sys
def lda_hyperparameter_generating_en(df_en, final_stop_words_en):
    #Remove stops words
    data_words_nostops_hf = stopwords_en(df_en['text'], final_stop_words_en)
    #Create the bigram from the non stop words
    data_words_bigram_hf = bigrams(data_words_nostops_hf)
    #Do lemmatization keeping only noun, adj, vb, adv, the lemmatization cuts off the ends of words so they can be
    #grouped an analyze better
    data_lemma_hf = data_lemmatization_en(data_words_bigram_hf, allowed_postags = ["NOUN", "ADJ", "VERB", "ADV"])
    #Create the dictionary, corpus, and term document matrix
    #Dictionary
    id2word_hf = corpora.Dictionary(data_lemma_hf)
    #Corpus
    texts_hf = data_lemma_hf
    #Term Document Matrix
    corpus_hf = [id2word_hf.doc2bow(text) for text in texts_hf]
    
    #Lets iterate over the function to find the optimal number for each of the hyper parameters
    grid_hf = {}
    grid_hf['Validation_Set'] = {}
    
    #Topic Range
    min_topics = 6
    max_topics = 8
    step_size = 1
    topic_range = range(min_topics, max_topics, step_size)
    
    #Alpha Parameter
    alpha = list(np.arange(0.01, 1, 0.3))
    alpha.append('symmetric')
    alpha.append('asymmetric')
    
    #Beta Parameter
    beta = list(np.arange(0.01, 1, 0.3))
    beta.append('symmetric')
    
    #Validation sets
    num_of_docs = len(corpus_hf)
    corpus_sets = [corpus_hf]
    corpus_title = ['100% Corpus']
    model_results = {'Validation_Set': [],
                     'Topics': [],
                     'Alpha': [],
                     'Beta': [],
                     'Coherence': []
                    }
    
    #iterate through validation corpora:
    for i in range(len(corpus_sets)):
        #iterate through number of topics:
        for k in topic_range:
            #iterate through alpha values:
            for a in alpha:
                #iterate through beta values:
                for b in beta:
                    #Get the coherence scores for the given hyperparameters
                    cv = compute_coherence_values(corpus = corpus_sets[i], texts = data_lemma_hf,
                                                  dictionary = id2word_hf, k = k, 
                                                  a = a, b = b)
                    #Save the Model Results 
                    model_results['Validation_Set'].append(corpus_title[i])
                    model_results['Topics'].append(k)
                    model_results['Alpha'].append(a)
                    model_results['Beta'].append(b)
                    model_results['Coherence'].append(cv)
    #Look at model results
    mr_en = pd.DataFrame(model_results)
    
    return mr_en

In [ ]:
#Step 20: Hyper Parameter Defining for English - getting the max coherence across the different amounts of topics
#Then getting the hyperparameters based on the max coherence value, (coherence is a value of topic interprebility, 
#a measure of semantic similarity
#https://stackoverflow.com/questions/20067636/pandas-dataframe-get-first-row-of-each-group
#https://stackoverflow.com/questions/10202570/find-row-where-values-for-column-is-maximal-in-a-pandas-dataframe
#https://stackoverflow.com/questions/15705630/get-the-rows-which-have-the-max-value-in-groups-using-groupby
#https://stackoverflow.com/questions/43193880/how-to-get-row-number-in-dataframe-in-pandas

def lda_hyper_define_en(mr_en):
    #Find the right number of topics
    mr2 = mr_en.groupby("Topics").max().reset_index()
    #Find the number of topics with the highest coherence
    max_coherence = mr2['Coherence'].max()
    mr3_5 = mr2.loc[mr2['Coherence'] == max_coherence]
    mr3 = mr3_5.reset_index(drop = True)
    #Get the Number of Topics for the highest coherence
    top_opt = mr3["Topics"][0]
    #Get the full data set of only the optimal number of topics
    mr_top_opt = mr_en['Topics'] == top_opt
    mr_to = mr_en[mr_top_opt]
    mr_to_2 = mr_to.reset_index(drop = True)
    #Get the hyperparameters for alpha and eta from mr_to_2 based on max coherence
    max_co_2 = mr_to_2['Coherence'].max()
    mr_to_3_5 = mr_to_2.loc[mr_to_2['Coherence'] == max_co_2]
    mr_to_3 = mr_to_3_5.reset_index(drop = True)
    #Convert mr_to_3, the optimal hyperparameters to a list 
    hyper_list_en = [mr_to_3["Topics"][0], mr_to_3["Alpha"][0], mr_to_3["Beta"][0]]
    print(hyper_list_en)
    return hyper_list_en

In [ ]:
#Step 21: Implement the stopwords, bigrams, and lemma functions English
def build_lda_en(df_en, final_stop_words_en, hyper_list_en):
    #Remove stops words
    data_words_nostops = stopwords_en(df_en['text'], final_stop_words_en)
    #Create the bigram from the non stop words
    data_words_bigram = bigrams(data_words_nostops)
    #Do lemmatization keeping only noun, adj, vb, adv, the lemmatization cuts off the ends of words so they can be
    #grouped an analyze better
    data_lemma = data_lemmatization_en(data_words_bigram, allowed_postags = ["NOUN", "ADJ", "VERB", "ADV"])
    #Create the dictionary, corpus, and term document matrix
    #Dictionary
    id2word = corpora.Dictionary(data_lemma)
    #Corpus
    texts = data_lemma
    #Term Document Matrix
    corpus = [id2word.doc2bow(text) for text in texts]
    #Train the actual LDA model
    #Watch out for too many topics
    lda_model_en = gensim.models.LdaMulticore(corpus = corpus,
                                              id2word = id2word,
                                              num_topics = hyper_list_en[0],
                                              random_state = 105,
                                              chunksize = 100,
                                              passes = 10,
                                              alpha = hyper_list_en[1],
                                              eta = hyper_list_en[2],
                                              per_word_topics = True,
                                              minimum_probability = 0)
    
    #Create the weights dataframe
    #Extract individual document topic proportions as determined by the LDA model. Our Gensim LDA model can classify 
    #the specific relative proportions for all ten topics within each document as long as you set the minimum_probability
    #argument to 0. If you did not do this, then some topics may be dropped from the final weighting if they did not 
    #meet the probability threshold set by default.
    weights_output = pd.DataFrame(columns = ['topic', 'prob_weight', 'doc_id'])
    
    #Extraction Loop: This loop extracts the topic proportions for all five topics for every individual document and
    #places them into a dataframe with a document-id key for merging topic proportion information with other datasets
    #about our corpus
    for i in range(0, len(corpus)):
        doc_weights = lda_model_en[corpus[i]][0]
        weights_df = pd.DataFrame(list(doc_weights), columns = ['topic', 'prob_weight'])
        weights_df['doc_id'] = i
        weights_output = weights_output.append(weights_df)
    
    #Create the daily (or hourly) weights data
    df2 = df_en
    df = weights_output
    
    #Create new dataset from the speechs with doc_id
    df3 = df2.reset_index()
    df3['doc_id'] = df3.index
    
    #Merge the Two Dataframe Together
    df4 = pd.merge(df, df3[['doc_id', 'Date', 'Year', 'Month', 'Day', 'Hour', 'Minute', 'Second', 'text']], on = 'doc_id', how = 'left')
    
    #Get the count of the total documents by Minute
    # This should be changed to Hour if I decide to do a full day of tweets instead
    total_docs = df4.groupby('Hour')['doc_id'].apply(lambda x: len(x.unique())).reset_index()
    
    #Label total_docs columns
    total_docs.columns = ['Hour', 'total_docs']
    
    #Get the Probability weight per Month and Topic 
    df_avg = df4.groupby(['Hour', 'topic']).agg({'prob_weight': 'sum'}).reset_index()
    
    #Combine the prob_weight and the total_docs data frames
    df_avg2 = df_avg.merge(total_docs, on = 'Hour', how = 'left')
    
    #Create the Average Weight of each Day and Topic
    df_avg2['average_weight'] = df_avg2['prob_weight'] / df_avg2['total_docs']
    
    #Get the Keywords from each Topics from the LDA Topic and Automatically Label them
    printtopics2 = lda_model_en.print_topics()
    lenpt2 = len(printtopics2)
    topic_label_list = []
    #For All the topics in generated by the model
    for i in range(0, lenpt2):
        pt_list = printtopics2[i][1].split('*')
        pt_list_words = []
        lenptl = len(pt_list)
        #Split the string list in the loop to get the first 5 topic words
        for j in range(1, 6):
            t_1 = pt_list[j]
            t_2 = t_1.split('+')
            t_3 = t_2[0]
            pt_list_words.append(t_3)
        topic_label_list.append(pt_list_words)
        
    #Set the Topic Labels to topic_label_list
    topic_labels = topic_label_list
    
    #Create topic_id numbers based on the createList function
    lenpt3 = len(printtopics2)
    topic_id = createList(lenpt3)
    
    #Combine the topic_id and topic_label
    data_tuple = list(zip(topic_id, topic_labels))
    
    #Convert into a dataframe
    df_labels = pd.DataFrame(data_tuple, columns = ['topic', 'topic_label'])
    
    #Merge labels into year weights data
    df_avg3 = df_avg2.merge(df_labels, on = 'topic')
    
    #Create the final per-document dataframe for broader analysis
    #Make sure to change on = ["Minute"] if want to use a different time scale
    df11_en = pd.merge(df4, df_avg3[['Hour', 'topic', 'average_weight', 'total_docs', 'topic_label']], 
                    on = ['Hour', 'topic'], how = 'left')
    
    return df11_en

In [ ]:
#Step 22: Visualization of Topics over Time English
#https://stackoverflow.com/questions/9622163/save-plot-to-image-file-instead-of-displaying-it-using-matplotlib
#https://stackoverflow.com/questions/12560600/creating-a-new-file-filename-contains-loop-variable-python
#https://stackoverflow.com/questions/33907776/how-to-create-an-array-of-dataframes-in-python
def viz_topic_time_en(df, hyper_list_en, counter_tm):
    
    #Split Data into individual topics
    topic_dfs_en = {}
    topic_label_list_en = []
    for i in range(0, hyper_list_en[0]):
        df_1_5 = df[df["topic"] == i]
        df_1 = df_1_5.reset_index(drop = True)
        topic_label_list_en.append(df_1["topic_label"][0])
        topic_plots = df_1.groupby("Hour")["average_weight"].mean()
        topic_dfs_en[i] = topic_plots
  
    #Change the size of the Plot
    plt.rcParams['figure.figsize'] = [20, 14]
    
    #Get the colors for the lines
    num_colors_en = hyper_list_en[0]
    
    color_en = ["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)])
                for i in range(0, num_colors_en)]
    
    #Create the plot
    #Change Legends based on Topic Labels, Plot the topic changes over time and colors
    for i in topic_dfs_en.keys():
        plt.plot(topic_dfs_en[i], color = color_en[i])
    plt.xlim(14, 20)
    plt.ylim(0, 1)
    plt.axhline(df['average_weight'].median(), color = "black")
    plt.title("Change in English Topics")
    plt.xlabel("Hour") 
    plt.ylabel("Average Hour Topic Weight")
    plt.legend((topic_label_list_en))
    plt.grid()
    plt.savefig("/Users/johnc.burns/Documents/Documents/PhD Year Three/My Paper 5/foo9/Output_Files/Topic_Model_Charts/Test_Output_EN_Hour_" + str(counter_tm))
    plt.close()

In [ ]:
#Part 23: The Main Function
def main_tm(counter_tm):
    tmfull = all_files(path_to_json)
    tmfull2 = lang_loop(tmfull)
    tmfull3 = date_func(tmfull2)
    tmfull4 = year_func(tmfull3)
    tmfull5 = month_func(tmfull4)
    tmfull6 = day_func(tmfull5)
    tmfull7 = hour_func(tmfull6)
    tmfull8 = minute_func(tmfull7)
    tmfull9 = second_func(tmfull8)
    eng_tweet = English_Tweets_Func(tmfull9)
    eng_t2 = sort_chrono(eng_tweet)
    final_stop_words_en = stopwords_en_func()
    mr_en = lda_hyperparameter_generating_en(eng_t2, final_stop_words_en)
    hyper_list_en = lda_hyper_define_en(mr_en)
    en_tm_final = build_lda_en(eng_t2, final_stop_words_en, hyper_list_en)
    viz_topic_time_en(en_tm_final, hyper_list_en, counter_tm)
    counter_tm = counter_tm + 1

In [ ]:
#Part 24: Adding initally sleep counter, need 1 hour in seconds to gather enough data for the first pass
#For second pass, it takes about 20 minutes for the program to run 
#For testing
from datetime import datetime
time.sleep(3600)

In [ ]:
#Step 25: Add the Runner 
#Add a Counter Token somewhere in the loop
if __name__ == "__main__":
    while(True):
        start = datetime.now()
        print(start)
        main_tm(counter_tm)
        end = datetime.now()
        print(end)
        print(end - start)
        counter_tm = counter_tm + 1
        time.sleep(2400)